# Experiment 12 — Theme Business Needs + Stage + L3 — Custom Prompt, IDs Only

Theme Business Needs + Value Stream Stage + base L3 candidates. Theme description, Epic context, hierarchy, ground truth, and model reasons are excluded.

## What the LLM sees

```text
task
theme
  └─ business_needs
value_stream_stage
  ├─ stage_id
  ├─ stage_name
  ├─ stage_description
  ├─ entrance_criteria
  └─ exit_criteria
candidate_l3_capabilities[]
  ├─ capability_id
  ├─ capability_name
  ├─ capability_description
  └─ capability_tier
```

**Not sent to the LLM:** Theme description, Epic description, Epic success criteria, L1/L2 hierarchy, ground truth, model reasons.


## Configuration and imports

In [ ]:
from pathlib import Path
from time import perf_counter
import ast
import io
import json
import os
import re
import tokenize

import httpx
import pandas as pd
from IPython.display import display

from common import (
    call_llm_with_metrics,
    load_gateway,
    parse_json_response,
    save_results_excel,
    score_sets,
)

NOTEBOOK_DIR = Path.cwd()
WORKSPACE_DIR = (
    NOTEBOOK_DIR.parent
    if (NOTEBOOK_DIR.parent / "epic_gen.csv").exists()
    else NOTEBOOK_DIR
)
DATA_DIR = Path(os.getenv("L3_EXPERIMENT_DATA_DIR", WORKSPACE_DIR))

THEME_PATH = Path(os.getenv("L3_THEME_PATH", DATA_DIR / "epic_gen.csv"))
STAGE_PATH = Path(os.getenv("L3_STAGE_PATH", DATA_DIR / "VSSrv.csv"))
STAGE_CAPABILITY_MAP_PATH = Path(
    os.getenv(
        "L3_STAGE_CAPABILITY_MAP_PATH",
        DATA_DIR / "VSSCaprv (1).csv",
    )
)
GROUND_TRUTH_PATH = Path(
    os.getenv(
        "L3_GROUND_TRUTH_PATH",
        NOTEBOOK_DIR / "epic_l3_ground_truth_all_themes.xlsx",
    )
)

# Run every Theme in epic_gen.csv. Ground truth is not consulted here.
THEME_IDS = (
    pd.read_csv(
        THEME_PATH,
        usecols=["key"],
        encoding="cp1252",
        encoding_errors="replace",
        dtype=str,
    )["key"]
    .dropna()
    .str.strip()
    .loc[lambda values: values.ne("")]
    .drop_duplicates()
    .head(20)
    .tolist()
)

VALUE_STREAM_STAGE_FIELD_ID = "customfield_18700"

# Optional single-example inspection. Leave None for batch execution only.
INSPECTION_THEME_ID = None
INSPECTION_EPIC_KEY = None

EXPERIMENT_NAME = "E12_BUSINESS_NEEDS_STAGE_CUSTOM_PROMPT"


## Retrieval

In [ ]:
def clean_text(v):
    if v is None or (not isinstance(v, (list, dict)) and pd.isna(v)):
        return ""
    return str(v).strip()

def parse_exported_list(v):
    if v is None or pd.isna(v):
        return []
    text = str(v).strip()
    if not text:
        return []
    if text.startswith("[") and text.endswith("]"):
        values = [ast.literal_eval(t.string) for t in tokenize.generate_tokens(io.StringIO(text).readline) if t.type == tokenize.STRING]
        if values:
            return [clean_text(x) for x in values]
    try:
        parsed = ast.literal_eval(text)
    except (SyntaxError, ValueError):
        return [text]
    return [clean_text(x) for x in parsed] if isinstance(parsed, (list, tuple, set)) else [clean_text(parsed)]

def read_table(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path, dtype=str, encoding="cp1252", encoding_errors="replace")
    return pd.read_excel(path, dtype=str)

def load_themes():
    frame = read_table(THEME_PATH)
    frame = frame.loc[frame["key"].isin(THEME_IDS)]
    themes = {}

    for _, row in frame.iterrows():
        epic_keys = parse_exported_list(row.get("epic_keys"))
        themes[clean_text(row["key"])] = {
            "theme_description": clean_text(row.get("description")),
            "theme_business_needs": clean_text(row.get("businessNeeds")),
            "epics": [{"key": epic_key} for epic_key in epic_keys],
        }

    return themes

def jira_headers():
    if os.getenv("JIRA_HEADERS_JSON"):
        return json.loads(os.environ["JIRA_HEADERS_JSON"])
    token = os.getenv("JIRA_BEARER_TOKEN") or os.getenv("JIRA_TOKEN")
    if token:
        return {"Authorization": f"Bearer {token}", "Accept": "application/json"}
    raise RuntimeError("Set JIRA_HEADERS_JSON, JIRA_BEARER_TOKEN, or JIRA_TOKEN.")

def epic_stage_ids(epic_key):
    url = f"{os.environ['JIRA_BASE_URL'].rstrip('/')}/rest/api/2/issue/{epic_key}"
    verify = os.getenv("JIRA_VERIFY_SSL", "false").lower() == "true"
    with httpx.Client(headers=jira_headers(), verify=verify, timeout=30) as client:
        response = client.get(url, params={"fields": VALUE_STREAM_STAGE_FIELD_ID})
        response.raise_for_status()
    raw = response.json().get("fields", {}).get(VALUE_STREAM_STAGE_FIELD_ID) or []
    raw = raw if isinstance(raw, list) else [raw]
    ids = []
    for value in raw:
        text = json.dumps(value, ensure_ascii=False) if isinstance(value, (dict, list)) else clean_text(value)
        ids.extend(re.findall(r"VSS\d+", text, flags=re.IGNORECASE))
    return list(dict.fromkeys(x.upper() for x in ids))

themes = load_themes()
stage_frame = read_table(STAGE_PATH)
stage_capability_map = read_table(STAGE_CAPABILITY_MAP_PATH)

def stage_context(stage_id):
    match = stage_frame.loc[stage_frame["Value Stream Stage ID"].astype(str).str.strip() == stage_id]
    if match.empty:
        raise KeyError(f"No stage metadata for {stage_id}")
    row = match.iloc[0]
    return {
        "stage_id": stage_id,
        "stage_name": clean_text(row["Value Stream Stage Name"]),
        "stage_description": clean_text(row["Value Stream Stage Description"]),
        "entrance_criteria": clean_text(row["Value Stream Stage Entrance Criteria"]),
        "exit_criteria": clean_text(row["Value Stream Stage Exit Criteria"]),
    }


## Candidate construction

In [ ]:
def candidate_rows_for_stage(stage_id):
    rows = stage_capability_map.loc[
        stage_capability_map["Value Stream Stage ID"].astype(str).str.strip()
        == stage_id
    ].copy()
    rows = (
        rows.drop_duplicates(subset=["Capability ID"], keep="first")
        .sort_values(["Capability Name", "Capability ID"], kind="stable")
    )

    return [
        {
            "capability_id": clean_text(row["Capability ID"]),
            "capability_name": clean_text(row["Capability Name"]),
            "capability_description": clean_text(row["Capability Description"]),
            "capability_tier": clean_text(row["Capability Tier"]),
        }
        for _, row in rows.iterrows()
    ]


## Production prompt

In [ ]:
SYSTEM_PROMPT = 'You are performing Level 3 business capability classification for one Epic.\n\nAn L3 capability is a Level 3 business capability: a specific business function within the enterprise capability hierarchy.\n\nUse the supplied Theme Business Needs and Value Stream Stage context to select the candidate L3 capabilities materially represented.\n\nEVIDENCE\n\nTheme Business Needs describes the business outcomes and needs the Theme is intended to address.\n\nValue Stream Stage defines the business activity boundary relevant to the Epic.\n\nFor each candidate L3:\n- capability_id is the exact identifier to return when the capability is selected.\n- capability_description is the primary semantic definition of the business function.\n- capability_name is the supporting business label.\n- capability_tier is supporting taxonomy context only.\n\nDo not infer meaning from capability_id.\n\nSELECTION RULES\n\nCompare the Theme Business Needs and Value Stream Stage context against the candidate L3 capability definitions.\n\nSelect every candidate L3 capability whose business function is directly supported by the supplied evidence.\n\nDo not select a capability merely because:\n- it belongs to the supplied Value Stream Stage,\n- it shares similar keywords,\n- it is generally related to the Theme,\n- it is adjacent, upstream, or downstream,\n- it provides data or support to another capability.\n\nOnly return capability_id values that exist in the supplied candidate list.\n\nIf none are supported, return an empty list.\n\nOUTPUT\n\nReturn JSON only:\n\n{"l3":["CAP00000123","CAP00000456"]}\n\nDo not return reasons, explanations, Markdown, or additional fields.'

def build_user_prompt(theme, epic, stage, candidate_rows):
    payload = {
        "task": "Select directly supported L3 business capability IDs from the supplied candidates.",
        "theme": {"business_needs": theme["theme_business_needs"]},
        "value_stream_stage": stage,
        "candidate_l3_capabilities": candidate_rows,
    }
    return json.dumps(payload, ensure_ascii=False, indent=2)


## Prediction

In [ ]:
def validate_l3_id_response(payload, candidate_ids):
    if set(payload) != {"l3"}:
        raise ValueError("LLM response must contain exactly one top-level field: l3.")
    raw = payload.get("l3")
    if not isinstance(raw, list):
        raise ValueError("LLM response must contain an 'l3' list.")
    allowed = {str(value).strip() for value in candidate_ids if str(value).strip()}
    selected = []
    seen = set()
    for index, capability_id in enumerate(raw, start=1):
        if not isinstance(capability_id, str):
            raise ValueError(f"L3 selection #{index} must be a capability_id string.")
        capability_id = capability_id.strip()
        if not capability_id:
            raise ValueError(f"L3 selection #{index} is empty.")
        if capability_id not in allowed:
            raise ValueError(f"LLM selected {capability_id}, which is not a supplied candidate.")
        if capability_id in seen:
            raise ValueError(f"LLM returned duplicate capability_id {capability_id}.")
        seen.add(capability_id)
        selected.append(capability_id)
    return selected


def predict_for_stage(gateway, theme, epic, stage_id):
    stage = stage_context(stage_id)
    candidates = candidate_rows_for_stage(stage_id)
    if not candidates:
        return {
            "stage": stage,
            "candidates": [],
            "user_prompt": build_user_prompt(theme, epic, stage, []),
            "raw_response": None,
            "selections": [],
            "metrics": None,
        }

    user_prompt = build_user_prompt(theme, epic, stage, candidates)
    raw_response, metrics = call_llm_with_metrics(
        gateway,
        SYSTEM_PROMPT,
        user_prompt,
    )
    selections = validate_l3_id_response(
        parse_json_response(raw_response),
        [candidate["capability_id"] for candidate in candidates],
    )
    return {
        "stage": stage,
        "candidates": candidates,
        "user_prompt": user_prompt,
        "raw_response": raw_response,
        "selections": selections,
        "metrics": metrics,
    }


def metric_text(value):
    return "n/a" if value is None else str(value)


def summarize_llm_calls(call_metrics):
    successful = call_metrics.loc[call_metrics["status"] == "ok"].copy()

    def numeric(column):
        return pd.to_numeric(successful[column], errors="coerce").dropna()

    latency = numeric("latency_seconds")
    input_tokens = numeric("input_tokens")
    output_tokens = numeric("output_tokens")
    total_tokens = numeric("total_tokens")

    return pd.DataFrame([{
        "successful_calls": len(successful),
        "failed_calls": int((call_metrics["status"] == "error").sum()),
        "usage_reported_calls": len(total_tokens),
        "avg_latency_seconds": float(latency.mean()) if len(latency) else None,
        "p50_latency_seconds": float(latency.quantile(0.50)) if len(latency) else None,
        "p95_latency_seconds": float(latency.quantile(0.95)) if len(latency) else None,
        "avg_input_tokens": float(input_tokens.mean()) if len(input_tokens) else None,
        "avg_output_tokens": float(output_tokens.mean()) if len(output_tokens) else None,
        "avg_total_tokens": float(total_tokens.mean()) if len(total_tokens) else None,
        "total_input_tokens": int(input_tokens.sum()) if len(input_tokens) else None,
        "total_output_tokens": int(output_tokens.sum()) if len(output_tokens) else None,
        "total_tokens": int(total_tokens.sum()) if len(total_tokens) else None,
    }])


def run_predictions(preflight):
    eligible_rows = preflight.loc[preflight["evaluation_eligible"]].copy()
    prediction_rows = []
    call_rows = []

    call_columns = [
        "experiment",
        "theme_id",
        "epic_key",
        "stage_id",
        "candidate_count",
        "status",
        "latency_seconds",
        "input_tokens",
        "output_tokens",
        "total_tokens",
        "selected_count",
        "error",
    ]

    print(
        f"\nRunning {EXPERIMENT_NAME}: "
        f"{len(eligible_rows)} preflight-valid Epics only"
    )

    if eligible_rows.empty:
        return (
            pd.DataFrame(),
            pd.DataFrame(columns=call_columns),
        )

    gateway = load_gateway()

    for epic_index, row in enumerate(
        eligible_rows.to_dict(orient="records"),
        start=1,
    ):
        theme_id = row["theme_id"]
        epic_key = row["epic_key"]
        stage_ids = json.loads(row["stage_ids"])
        theme = themes[theme_id]
        epic = next(
            item for item in theme["epics"] if item["key"] == epic_key
        )

        stage_predictions = []
        predicted_ids = set()
        status = "ok"
        error = None

        print(
            f"\n[LLM {epic_index}/{len(eligible_rows)}] "
            f"{theme_id} | {epic_key}"
        )

        for stage_id in stage_ids:
            started = perf_counter()
            try:
                result = predict_for_stage(
                    gateway,
                    theme,
                    epic,
                    stage_id,
                )
                candidates = result["candidates"]

                if not candidates:
                    print(f"  {stage_id} SKIP | no candidates")
                    continue

                metrics = result["metrics"]
                selected_ids = result["selections"]
                print(
                    f"  {stage_id} OK"
                    f" | candidates={len(candidates)}"
                    f" | latency={metrics['latency_seconds']:.3f}s"
                    f" | input_tokens={metric_text(metrics['input_tokens'])}"
                    f" | output_tokens={metric_text(metrics['output_tokens'])}"
                    f" | total_tokens={metric_text(metrics['total_tokens'])}"
                    f" | selected={selected_ids}"
                )

                call_rows.append({
                    "experiment": EXPERIMENT_NAME,
                    "theme_id": theme_id,
                    "epic_key": epic_key,
                    "stage_id": stage_id,
                    "candidate_count": len(candidates),
                    "status": "ok",
                    "latency_seconds": metrics["latency_seconds"],
                    "input_tokens": metrics["input_tokens"],
                    "output_tokens": metrics["output_tokens"],
                    "total_tokens": metrics["total_tokens"],
                    "selected_count": len(selected_ids),
                    "error": None,
                })
                stage_predictions.append({
                    "stage_id": stage_id,
                    "selected_l3_ids": selected_ids,
                })
                predicted_ids.update(selected_ids)
            except Exception as exc:
                latency = perf_counter() - started
                status = "error"
                error = str(exc)
                print(
                    f"  {stage_id} ERROR"
                    f" | latency={latency:.3f}s"
                    f" | {error}"
                )
                call_rows.append({
                    "experiment": EXPERIMENT_NAME,
                    "theme_id": theme_id,
                    "epic_key": epic_key,
                    "stage_id": stage_id,
                    "candidate_count": None,
                    "status": "error",
                    "latency_seconds": latency,
                    "input_tokens": None,
                    "output_tokens": None,
                    "total_tokens": None,
                    "selected_count": None,
                    "error": error,
                })
                break

        prediction_rows.append({
            "experiment": EXPERIMENT_NAME,
            "theme_id": theme_id,
            "epic_key": epic_key,
            "stage_ids": row["stage_ids"],
            "ground_truth_l3_ids": row["ground_truth_l3_ids"],
            "available_candidate_l3_ids": row["available_candidate_l3_ids"],
            "gt_found_in_candidates": row["gt_found_in_candidates"],
            "gt_missing_from_candidates": row["gt_missing_from_candidates"],
            "predicted_l3_ids": json.dumps(sorted(predicted_ids)),
            "stage_predictions": json.dumps(
                stage_predictions,
                ensure_ascii=False,
            ),
            "status": status,
            "error": error,
        })

    return (
        pd.DataFrame(prediction_rows),
        pd.DataFrame(call_rows, columns=call_columns),
    )


## Single-example inspection

In [ ]:
if INSPECTION_THEME_ID and INSPECTION_EPIC_KEY:
    theme = themes[INSPECTION_THEME_ID]
    epic = next(
        item
        for item in theme["epics"]
        if item["key"] == INSPECTION_EPIC_KEY
    )
    stage_id = epic_stage_ids(epic["key"])[0]
    stage = stage_context(stage_id)
    candidates = candidate_rows_for_stage(stage_id)
    user_prompt = build_user_prompt(theme, epic, stage, candidates)

    print("SYSTEM PROMPT")
    print(SYSTEM_PROMPT)
    print("\nUSER PROMPT")
    print(user_prompt)
    print("\nCANDIDATES")
    display(pd.DataFrame(candidates))

    if candidates:
        result = predict_for_stage(
            load_gateway(),
            theme,
            epic,
            stage_id,
        )
        print("\nMODEL RESPONSE")
        print(result["raw_response"])
        print("\nCALL METRICS")
        display(pd.DataFrame([result["metrics"]]))
    else:
        print("\nNo candidates for this Stage; LLM call skipped.")
else:
    print(
        "Set INSPECTION_THEME_ID and INSPECTION_EPIC_KEY "
        "to inspect one example."
    )


## Batch preflight, execution, and evaluation

Ground truth is loaded **before any LLM call** only to validate the experiment population. An Epic is sent to the LLM only when GT exists, a Stage exists, candidates exist, and every GT L3 is present in the union of Stage candidates. GT is never included in the model prompt.


In [ ]:
def ground_truth_by_epic():
    gt = read_table(GROUND_TRUTH_PATH).copy()
    gt["l3_capability_id"] = (
        gt["l3_capability_id"]
        .fillna("")
        .astype(str)
        .str.strip()
    )
    gt = gt.loc[gt["l3_capability_id"].ne("")]
    return {
        key: set(group["l3_capability_id"])
        for key, group in gt.groupby("epic_key", sort=False)
    }


def preflight_population():
    truth = ground_truth_by_epic()
    rows = []
    total_epics = sum(len(theme["epics"]) for theme in themes.values())
    check_index = 0

    print("\n================ PRECHECK ================")
    print(f"Themes selected: {len(themes)}")
    print(f"Total Epics: {total_epics}")

    for theme_id, theme in themes.items():
        for epic in theme["epics"]:
            check_index += 1
            epic_key = epic["key"]
            gt_ids = truth.get(epic_key)
            stage_ids = []
            candidate_ids = set()
            reason = ""
            error = None

            print(
                f"\n[GT CHECK {check_index}/{total_epics}] "
                f"{theme_id} | {epic_key}"
            )

            if gt_ids is None:
                reason = "missing_ground_truth"
            else:
                try:
                    stage_ids = epic_stage_ids(epic_key)
                except Exception as exc:
                    reason = "error"
                    error = str(exc)

                if not reason and not stage_ids:
                    reason = "no_stage"

                if not reason:
                    try:
                        for stage_id in stage_ids:
                            # Validate stage metadata now so invalid rows never
                            # reach the LLM phase.
                            stage_context(stage_id)
                            candidates = candidate_rows_for_stage(stage_id)
                            candidate_ids.update(
                                candidate["capability_id"]
                                for candidate in candidates
                            )
                    except Exception as exc:
                        reason = "error"
                        error = str(exc)

                if not reason and not candidate_ids:
                    reason = "no_candidates"

            if gt_ids is None:
                gt_found = set()
                gt_missing = set()
            else:
                gt_found = gt_ids & candidate_ids
                gt_missing = gt_ids - candidate_ids

            if not reason and gt_missing:
                reason = "gt_not_fully_retrievable"

            eligible = gt_ids is not None and not reason

            print(f"Stages: {stage_ids}")
            print(f"GT L3s: {sorted(gt_ids) if gt_ids is not None else []}")
            print(f"Candidate L3s: {sorted(candidate_ids)}")
            print(f"GT found in candidates: {sorted(gt_found)}")
            print(f"GT missing from candidates: {sorted(gt_missing)}")

            if eligible:
                print("STATUS: VALID")
            else:
                detail = f" | {error}" if error else ""
                print(f"STATUS: INVALID - {reason}{detail}")

            rows.append({
                "experiment": EXPERIMENT_NAME,
                "theme_id": theme_id,
                "epic_key": epic_key,
                "stage_ids": json.dumps(stage_ids),
                "ground_truth_l3_ids": (
                    json.dumps(sorted(gt_ids))
                    if gt_ids is not None
                    else None
                ),
                "available_candidate_l3_ids": json.dumps(
                    sorted(candidate_ids)
                ),
                "gt_found_in_candidates": json.dumps(sorted(gt_found)),
                "gt_missing_from_candidates": json.dumps(
                    sorted(gt_missing)
                ),
                "evaluation_eligible": eligible,
                "evaluation_exclusion_reason": reason,
                "preflight_error": error,
            })

    preflight = pd.DataFrame(rows)
    valid_count = int(preflight["evaluation_eligible"].sum())
    invalid_count = len(preflight) - valid_count

    print("\n================ PRECHECK SUMMARY ================")
    print(f"Themes selected: {len(themes)}")
    print(f"Total Epics: {len(preflight)}")
    print(f"VALID Epics: {valid_count}")
    print(f"INVALID Epics: {invalid_count}")
    print("Invalid breakdown:")
    for reason in (
        "missing_ground_truth",
        "no_stage",
        "no_candidates",
        "gt_not_fully_retrievable",
        "error",
    ):
        count = int(
            (preflight["evaluation_exclusion_reason"] == reason).sum()
        )
        print(f"  {reason}: {count}")
    print(f"LLM calls will run ONLY for: {valid_count} Epics")
    print("==================================================")

    return preflight


def evaluate_predictions(prediction_frame):
    out = []
    for row in prediction_frame.to_dict(orient="records"):
        pred = set(json.loads(row["predicted_l3_ids"]))
        gt = set(json.loads(row["ground_truth_l3_ids"]))

        if row["status"] == "error":
            metrics = {
                "exact_match": None,
                "precision": None,
                "recall": None,
                "f1": None,
                "predicted_count": len(pred),
                "truth_count": len(gt),
            }
        else:
            metrics = score_sets(pred, gt)

        row.update(metrics)
        out.append(row)

    return pd.DataFrame(out)


def evaluation_summary(results, preflight):
    scored = results.loc[results["exact_match"].notna()]
    valid_preflight = int(preflight["evaluation_eligible"].sum())

    summary = pd.DataFrame([{
        "scope": "valid_evaluation_population",
        "evaluated_epics": len(scored),
        "exact_match_accuracy": (
            scored["exact_match"].mean() if len(scored) else 0.0
        ),
        "mean_precision": (
            scored["precision"].mean() if len(scored) else 0.0
        ),
        "mean_recall": (
            scored["recall"].mean() if len(scored) else 0.0
        ),
        "mean_f1": scored["f1"].mean() if len(scored) else 0.0,
    }])

    diagnostics = pd.DataFrame([{
        "themes_selected": len(themes),
        "total_epics": len(preflight),
        "preflight_valid_epics": valid_preflight,
        "preflight_invalid_epics": len(preflight) - valid_preflight,
        "missing_ground_truth": int((
            preflight["evaluation_exclusion_reason"]
            == "missing_ground_truth"
        ).sum()),
        "no_stage": int((
            preflight["evaluation_exclusion_reason"] == "no_stage"
        ).sum()),
        "no_candidates": int((
            preflight["evaluation_exclusion_reason"] == "no_candidates"
        ).sum()),
        "gt_not_fully_retrievable": int((
            preflight["evaluation_exclusion_reason"]
            == "gt_not_fully_retrievable"
        ).sum()),
        "preflight_errors": int((
            preflight["evaluation_exclusion_reason"] == "error"
        ).sum()),
        "llm_prediction_errors": int((
            results["status"] == "error"
        ).sum()) if len(results) else 0,
        "scored_epics": len(scored),
    }])

    return summary, diagnostics


# Ground truth is used here only to validate the experiment population.
# It is never included in the LLM prompt.
preflight = preflight_population()
predictions, llm_calls = run_predictions(preflight)
results = evaluate_predictions(predictions)
summary, diagnostics = evaluation_summary(results, preflight)
llm_call_summary = summarize_llm_calls(llm_calls)

print("\nEvaluation summary")
display(summary)
print("\nPreflight diagnostics")
display(diagnostics)
print("\nLLM latency / token summary")
display(llm_call_summary)
print("\nPer-call LLM metrics")
display(llm_calls.head(50))
if len(results):
    display(results.head(20))

output_path = save_results_excel(
    results,
    EXPERIMENT_NAME,
    "results",
    extra_sheets={
        "evaluation_summary": summary,
        "preflight": preflight,
        "diagnostics": diagnostics,
        "llm_calls": llm_calls,
        "llm_call_summary": llm_call_summary,
    },
)
print(f"Saved {output_path}")
